# 05 · Robustness

Four questions: does the result survive a stricter tail definition, is the close
actually different from any other half hour, is it stable through time, and does
it depend on market regime.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 60)
from closingbell import config as C, calendar_utils as cal

def table(name):
    return pd.read_csv(C.TABLES / f"{name}.csv")

sessions = pd.read_parquet(C.PROCESSED / "sessions.parquet")
print(f"{len(sessions):,} ticker-sessions, {sessions.session.min()} to {sessions.session.max()}")

15,756 ticker-sessions, 2021-01-04 to 2026-03-31


## 1% tails instead of 5%

In [2]:
t = table('central_2x2_overnight_1pct')
t[["direction", "volume_regime", "n", "mean_close30_bps", "mean_bps",
   "excess_bps", "excess_ci_low_bps", "excess_ci_high_bps", "excess_p_value"]].round(2)

,direction,volume_regime,n,mean_close30_bps,mean_bps,excess_bps,excess_ci_low_bps,excess_ci_high_bps,excess_p_value
0,all (baseline),all,14958,-0.16,4.81,NaN,NaN,NaN,NaN
1,strong up,high,216,108.19,6.80,1.99,-28.84,32.05,0.88
2,strong up,ordinary,55,75.71,2.66,-2.15,-34.54,32.87,0.89
3,strong down,high,191,-113.17,19.24,14.43,-17.93,45.61,0.37
4,strong down,ordinary,46,-104.68,49.64,44.83,-14.38,98.94,0.14


Same qualitative picture with much wider intervals; no cell's excess over the drift is distinguishable from zero at the 1% cut.

## Is the close special?

Three non-overlapping 30-minute windows, each standardised within its own slot,
each tested against the period that follows. This is the placebo the closing
effect has to beat.

In [3]:
tod = table('time_of_day_regressions')
tod[["window", "predicts", "beta_std", "se", "t", "p", "ci_low", "ci_high", "n"]].round(4)

,window,predicts,beta_std,se,t,p,ci_low,ci_high,n
0,morning 09:30-10:00,10:00-10:30,0.0051,0.0149,0.3393,0.7344,-0.0242,0.0343,15144
1,midday 12:00-12:30,12:30-13:00,-0.0504,0.0290,-1.7364,0.0825,-0.1073,0.0065,15136
2,close 15:30-16:00,overnight,-0.0533,0.0198,-2.6974,0.0070,-0.0920,-0.0146,14999


The closing coefficient (−0.053) is the only one significant on its own, but it
is **statistically indistinguishable from the midday coefficient** (−0.050) —
the intervals overlap almost entirely. The close's edge is a tighter standard
error, not a bigger effect.

In [4]:
print(table('time_of_day_persistence').round(3).to_string(index=False))
print()
print("raw scale of each window (the reason the comparison is run standardised):")
print(table('time_of_day_scale').round(2).to_string(index=False))

             window    predicts    n  p_same_sign  ci_low  ci_high  mean_next_bps
morning 09:30-10:00 10:00-10:30 2052        0.502   0.471    0.532         -0.915
 midday 12:00-12:30 12:30-13:00 1993        0.479   0.441    0.518          1.874
  close 15:30-16:00   overnight 1975        0.463   0.424    0.503         15.107

raw scale of each window (the reason the comparison is run standardised):
             window  sd_window_bps  mean_abs_window_bps   following  sd_following_bps     n
morning 09:30-10:00          92.02                64.42 10:00-10:30             60.47 15700
 midday 12:00-12:30          36.52                24.97 12:30-13:00             33.59 15676
  close 15:30-16:00          41.09                28.44   overnight            138.00 15568


In [5]:
table('time_of_day_regressions_extremes').round(4)

,window,predicts,beta_std,se,t,p,ci_low,ci_high,n,r2,sample
0,morning 09:30-10:00,10:00-10:30,-0.0081,0.0183,-0.4455,0.6560,-0.0439,0.0276,2052,0.0002,extremes
1,midday 12:00-12:30,12:30-13:00,-0.0549,0.0412,-1.3300,0.1835,-0.1357,0.0260,1993,0.0074,extremes
2,close 15:30-16:00,overnight,-0.0547,0.0247,-2.2103,0.0271,-0.1032,-0.0062,1974,0.0067,extremes


Restricting to extreme moves in each window tells the same story with less precision.

See `results/figures/fig09`.

## Stability through time

In [6]:
roll = table('rolling_relationship')
roll["end_session"] = pd.to_datetime(roll.end_session)
print("rolling 252-session slope: min %.3f, max %.3f" % (roll.coef.min(), roll.coef.max()))
print("share of windows whose 95%% band excludes zero: %.1f%%" %
      (100 * ((roll.ci_high < 0) | (roll.ci_low > 0)).mean()))
roll.set_index("end_session")[["coef", "ci_low", "ci_high"]].iloc[::80].round(3)

rolling 252-session slope: min -0.471, max 0.066
share of windows whose 95% band excludes zero: 20.0%


,coef,ci_low,ci_high
end_session,,,
2022-03-02,0.031,-0.228,0.290
2022-06-27,0.023,-0.259,0.304
2022-10-25,-0.065,-0.332,0.201
2023-02-22,-0.116,-0.428,0.196
2023-06-22,-0.135,-0.472,0.203
2023-10-17,-0.273,-0.572,0.026
2024-02-13,-0.188,-0.450,0.073
2024-06-07,-0.068,-0.315,0.178
2024-10-03,-0.341,-0.592,-0.090


The relationship is near zero at both ends of the sample and strongly negative in the middle. A full-sample point estimate averages over regimes that look genuinely different.

See `results/figures/fig08`.

## Regimes

Exploratory. With seven regime variables and two directions, several
individually significant cells are expected under the null.

In [7]:
rg = table('regime_tables')
rg[["regime_var", "regime", "direction", "n", "mean_bps", "ci_low_bps",
    "ci_high_bps", "t_stat", "p_same_sign"]].round(2)

,regime_var,regime,direction,n,mean_bps,ci_low_bps,ci_high_bps,t_stat,p_same_sign
0,vix_regime,high VIX,strong up,544,8.08,-16.26,33.34,-0.93,0.56
1,vix_regime,high VIX,strong down,597,20.69,-5.89,46.19,2.19,0.42
2,vix_regime,low VIX,strong up,472,2.61,-13.13,19.08,-0.12,0.50
3,vix_regime,low VIX,strong down,361,32.75,11.18,54.80,2.50,0.34
4,market_day,flat,strong up,460,1.28,-17.65,20.35,-0.92,0.50
5,market_day,flat,strong down,416,30.82,10.05,49.92,2.59,0.34
6,market_day,strong down,strong up,189,-15.53,-50.90,14.65,-0.68,0.52
7,market_day,strong down,strong down,438,11.00,-22.44,41.22,1.24,0.45
8,market_day,strong up,strong up,367,21.73,-7.52,51.87,0.29,0.58
9,market_day,strong up,strong down,104,62.85,17.32,116.15,2.05,0.37


In [8]:
down = rg[rg.direction == "strong down"]
print("reversal after down closes, by regime (mean overnight, bp):")
down.sort_values("mean_bps", ascending=False)[["regime_var", "regime", "n", "mean_bps"]].round(1).to_string(index=False)

reversal after down closes, by regime (mean overnight, bp):


'    regime_var      regime   n  mean_bps\n    market_day   strong up 104      62.8\n           dow     Tuesday 174      57.1\n  is_month_end        True  98      42.6\n           dow    Thursday 218      37.9\n    vix_regime     low VIX 361      32.7\n    market_day        flat 416      30.8\nis_quarter_end        True  37      29.6\n volume_regime        high 611      29.3\n       is_opex       False 921      25.3\nis_quarter_end       False 921      25.1\n  is_month_end       False 860      23.3\n       is_opex        True  37      23.2\n           dow      Monday 134      23.2\n    vix_regime    high VIX 597      20.7\n           dow   Wednesday 260      19.6\n volume_regime    ordinary 347      18.0\n    market_day strong down 438      11.0\n           dow      Friday 172     -13.1'

Reversal after down closes appears in both VIX regimes and is largest when the
market itself rose that day, and at month end. Day-of-week differences are almost
certainly noise.

## Other outcome horizons

In [9]:
oo = table('regression_other_outcomes')
oo[oo.term.isin(["r_close", "avol", "r_close_x_avol"])][["outcome", "term", "coef", "se", "t", "p"]].round(4)

,outcome,term,coef,se,t,p
1,r_open30_next,r_close,0.0489,0.0353,1.3821,0.1669
2,r_open30_next,avol,1.8830,3.2799,0.5741,0.5659
3,r_open30_next,r_close_x_avol,-0.1769,0.0498,-3.5539,0.0004
13,r_session_next,r_close,0.1667,0.0865,1.9270,0.0540
14,r_session_next,avol,-0.4128,6.5914,-0.0626,0.9501
15,r_session_next,r_close_x_avol,-0.2857,0.0988,-2.8915,0.0038
25,r_next_close_to_close,r_close,0.0249,0.1143,0.2176,0.8277
26,r_next_close_to_close,avol,9.1470,7.0217,1.3027,0.1927
27,r_next_close_to_close,r_close_x_avol,-0.2920,0.1147,-2.5467,0.0109


## Leakage

Every event flag is built from strictly earlier sessions. The claim is tested
directly in `tests/test_no_leakage_end_to_end.py`: run the pipeline on the full
sample and on a truncated sample, and every classification on the overlap must be
identical.

See `docs/data_leakage_audit.md`.

In [10]:
ev_rows = sessions[sessions.is_extreme.fillna(False)]
print("minimum trailing observations behind any event flag:")
print("  closing z-score:", int(ev_rows.close30_n.min()))
print("  AVOL median    :", int(ev_rows.cvol_n.min()))
print("  configured floor:", C.MIN_TRAILING_OBS)

minimum trailing observations behind any event flag:
  closing z-score: 40
  AVOL median    : 40
  configured floor: 40
